In [20]:
# Librerías para manejar rutas y archivos
from pathlib import Path

# Librería para generar los conjuntos de puntos
from sklearn.datasets import make_blobs

# Librerías para ejecutar los programas de C++ y analizar resultados
import subprocess
import pandas as pd
import numpy as np
import os

# Buscar la carpeta principal del proyecto
def find_project_root(start=None):

    # Obtener la carpeta desde donde estamos ejecutando el notebook
    current = Path(start or Path.cwd()).resolve()

    # Buscar la carpeta que contiene .git
    for folder in [current, *current.parents]:
        if (folder / ".git").exists():
            return folder
        
    raise FileNotFoundError("No se encontró la raíz del proyecto")

# Guardar la ruta principal del proyecto
PROJECT_ROOT = find_project_root()

# Carpeta donde se encuentran los datos de entrada
INPUT_DIR = PROJECT_ROOT / "data" / "input"

# Carpetas donde cada programa de C++ guarda sus resultados
OUTPUT_DIR_SERIAL = PROJECT_ROOT / "data" / "output" / "serial"
OUTPUT_DIR_PARALLEL = PROJECT_ROOT / "data" / "output" / "parallel_full"
OUTPUT_DIR_PARALLEL_K = PROJECT_ROOT / "data" / "output" / "parallel_k"

# Carpeta donde se encuentran los programas de C++
SRC_DIR = PROJECT_ROOT / "src"

# Carpeta donde guardaremos los resultados de los experimentos
RESULTADOS_DIR = PROJECT_ROOT / "resultados"

# Crear las carpetas en caso de que no existan
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_SERIAL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_PARALLEL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_PARALLEL_K.mkdir(parents=True, exist_ok=True)
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
# Indicate the number of points
n_points = [20000, 40000, 80000, 120000, 140000, 160000, 180000, 200000]

for n in n_points:
    # randomly generating data points and noise
    points, y_true = make_blobs(n_samples=n,
                                centers=4,
                                cluster_std=0.06,
                                random_state=11,
                                center_box=(0, 1.0))

    # only positive points and with three decimals
    points = np.round(np.abs(points[:, ::-1]), 3)

    input_file = INPUT_DIR / f"{n}_data.csv"

    # storing points into a csv file
    np.savetxt(input_file, points, delimiter=",",  fmt="%.3f")

In [22]:
# Distancia máxima para considerar que dos puntos son vecinos
epsilon = 0.03

# Número mínimo de puntos para considerar que un punto es core
# Se incluye al propio punto
min_puntos = 10

# Obtener el número de procesadores lógicos disponibles
cores_virtuales = os.cpu_count()

# Cantidades de hilos que utilizaremos en los experimentos
# 1 hilo, mitad de los procesadores, todos y el doble
num_hilos = [1, cores_virtuales//2, cores_virtuales, cores_virtuales*2]

# Cantidades de particiones para parallel_k
valores_k = [2, 4, 8, 16, 32, 64]

# Cantidades de puntos que utilizaremos en las pruebas
n_points = [20000, 40000, 80000, 120000, 140000, 160000, 180000, 200000]

# Número de veces que repetiremos cada configuración
num_pruebas = 10

In [25]:
# Ejecutar uno de los programas de C++ y obtener su tiempo
def ejecutar_programa(programa, n, epsilon, min_puntos, num_hilos=None, k=None, num_pruebas=None):

    # Construir el comando que utilizaremos para ejecutar el programa
    # Los tres programas reciben n, epsilon y min_puntos
    comando = [
        str(SRC_DIR / programa),
        str(n),
        str(epsilon),
        str(min_puntos)
    ]

    # El programa serial no recibe número de hilos
    # parallel_full y parallel_k sí lo reciben
    if num_hilos is not None:
        comando.append(str(num_hilos))

    # Solamente parallel_k recibe el número de particiones
    if k is not None:
        comando.append(str(k))

    # parallel_k recibe el número de repeticiones
    if programa == "parallel_k" and num_pruebas is not None:
        comando.append(str(num_pruebas))

    # Ejecutar el programa de C++
    # cwd indica que se ejecutará desde la raíz del proyecto
    # capture_output guarda lo que el programa imprime en la terminal
    resultado = subprocess.run(
        comando,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    # Verificar si el programa terminó correctamente
    if resultado.returncode != 0:

        # Mostrar los mensajes del programa en caso de error
        print(resultado.stdout)
        print(resultado.stderr)

        raise RuntimeError(
            f"Error al ejecutar {programa} con n={n}, hilos={num_hilos}, k={k}"
        )
    
    # Buscar el tiempo dentro de lo que imprimió el programa
    for linea in resultado.stdout.splitlines():

        # Comprobar si la línea comienza con la etiqueta que buscamos
        if linea.startswith("TIEMPO="):
            # Separar el texto por el signo "=" y convertir el tiempo a float
            return float(linea.split("=")[1])
        
    # Si el programa terminó pero no encontramos el tiempo
    raise RuntimeError(
        f"No se encontró TIEMPO= en la salida de {programa}"
    )

In [26]:
# Lista donde guardaremos los resultados de todas las configuraciones
resultados = []

# Nombres de las columnas del DataFrame y del CSV
columnas = [
    "n_puntos",
    "hilos",
    "k",
    "tiempo_serial",
    "tiempo_parallel_full",
    "tiempo_parallel_k",
    "speedup_full",
    "speedup_k"
]

# Recorrer los diferentes tamaños de datos
for n in n_points:

    # Crear la ruta del CSV correspondiente a esta cantidad de puntos
    CSV_RESULTADOS = RESULTADOS_DIR / f"resultados_{n}.csv"

    # Si el CSV existe y tiene todas las configuraciones, no repetir experimentos
    total_esperado = len(num_hilos) * len(valores_k)

    if CSV_RESULTADOS.exists():

        df_existente = pd.read_csv(CSV_RESULTADOS)

        # Comprobar que se hayan guardado todas las configuraciones
        configuraciones = df_existente[["hilos", "k"]].drop_duplicates()

        if (
            len(configuraciones) == total_esperado
            and df_existente["n_puntos"].eq(n).all()
        ):
            print(f"\nSaltando n = {n}: resultados completos.")
            continue

    # Reiniciar los resultados para esta cantidad de puntos
    resultados = []

    print(f"\nEjecutando n = {n}")

    # -----------------------------------
    # Versión serial
    # -----------------------------------

    # Guardar los tiempos de cada repetición serial
    tiempo_serial = []

    # Ejecutar la versión serial num_pruebas veces
    for prueba in range(num_pruebas):

        tiempo = ejecutar_programa(
            "serial",
            n,
            epsilon,
            min_puntos
        )

        # Guardar el tiempo de esta repetición
        tiempo_serial.append(tiempo)

    # Calcular el tiempo promedio de la versión serial
    tiempo_serial_promedio = sum(tiempo_serial) / num_pruebas

    # Recorrer las diferentes cantidades de hilos
    for hilos in num_hilos:

        # -----------------------------------
        # Primera versión paralela
        # -----------------------------------

        # Guardar los tiempos de cada repetición
        tiempo_parallel = []

        # Ejecutar parallel_full num_pruebas veces
        for prueba in range(num_pruebas):

            tiempo = ejecutar_programa(
                "parallel_full",
                n,
                epsilon,
                min_puntos,
                hilos
            )

            tiempo_parallel.append(tiempo)

        # Calcular el tiempo promedio de parallel_full
        tiempo_parallel_full_promedio = (
            sum(tiempo_parallel) / num_pruebas
        )

        # Calcular el speedup respecto a la versión serial
        speedup_full = (
            tiempo_serial_promedio / tiempo_parallel_full_promedio
        )

        # -----------------------------------
        # Segunda versión paralela
        # -----------------------------------

        # Probar las diferentes cantidades de particiones
        for k in valores_k:

            # Ejecutar parallel_k con todas las repeticiones dentro de C++
            # Las particiones se construyen una sola vez por ejecución
            tiempo_parallel_k_promedio = ejecutar_programa(
                "parallel_k",
                n,
                epsilon,
                min_puntos,
                hilos,
                k,
                num_pruebas
            )

            # Calcular el speedup respecto a la versión serial
            speedup_k = (
                tiempo_serial_promedio / tiempo_parallel_k_promedio
            )

            # Guardar los resultados de la configuración actual
            resultados.append({
                "n_puntos": n,
                "hilos": hilos,
                "k": k,
                "tiempo_serial": tiempo_serial_promedio,
                "tiempo_parallel_full": tiempo_parallel_full_promedio,
                "tiempo_parallel_k": tiempo_parallel_k_promedio,
                "speedup_full": speedup_full,
                "speedup_k": speedup_k
            })

            # Crear el DataFrame con los resultados obtenidos hasta ahora
            df_resultados = pd.DataFrame(
                resultados,
                columns=columnas
            )

            # Guardar los resultados en el CSV
            # Se actualiza después de cada configuración
            df_resultados.to_csv(
                CSV_RESULTADOS,
                index=False
            )

# Mostrar mensaje cuando terminen todos los experimentos
print("\nExperimentos finalizados")

# Mostrar la ubicación del CSV
print(f"Resultados guardados en: {RESULTADOS_DIR}")


Ejecutando n = 20000

Ejecutando n = 40000

Ejecutando n = 80000


KeyboardInterrupt: 

In [ ]:
# Leer los archivos de resultados de cada cantidad de puntos
archivos = sorted(RESULTADOS_DIR.glob("resultados_*.csv"))

# Unir los resultados en un solo DataFrame
df_resultados = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos],
    ignore_index=True
)

# Mostrar los resultados
print(f"Total de configuraciones: {len(df_resultados)}")

display(df_resultados)